In [12]:
import pandas as pd
import glob
import os

In [13]:
folder_path = 'data_mentah'
nama_file_txt = 'udemy_list_topics.txt'

In [14]:
df_list = []

if os.path.exists(folder_path):
    file_list = glob.glob(os.path.join(folder_path, "*.csv"))
    print(f"Ditemukan {len(file_list)} file CSV. Mulai membaca...")

    for file in file_list:
        try:
            df = pd.read_csv(file)
            df_list.append(df)
        except Exception as e:
            print(f"Gagal membaca file {os.path.basename(file)}: {e}")
else:
    print(f"Error: Folder '{folder_path}' tidak ditemukan.")

# Menggabungkan semua dataframe
if len(df_list) > 0:
    df_gabungan = pd.concat(df_list, ignore_index=True)
    print(f"Berhasil digabungkan! Dimensi awal: {df_gabungan.shape[0]} baris, {df_gabungan.shape[1]} kolom")
else:
    print("Dataframe gagal dibentuk. Periksa kembali folder dan file Anda.")

Ditemukan 102 file CSV. Mulai membaca...
Berhasil digabungkan! Dimensi awal: 95700 baris, 16 kolom


In [15]:
print("Missing values sebelum pembersihan:")
print(df_gabungan.isnull().sum())
print(f"\nDimensi awal: {df_gabungan.shape}")

Missing values sebelum pembersihan:
course_title                 0
course_id                   40
lecturer_name              194
related_topics              40
subject                      0
level                       27
durations                 8473
ratings                   1524
num_ratings               6442
student                   2720
price                      700
last_update                200
articles                     0
exercise                     0
downloadable_resources       0
url                          0
dtype: int64

Dimensi awal: (95700, 16)


In [16]:
kolom_wajib = ['course_title', 'course_id', 'lecturer_name', 'subject', 'level',
               'related_topics', 'price', 'last_update', 'url']
df_gabungan.dropna(subset=kolom_wajib, inplace=True)

kolom_numerik_nullable = ['durations', 'ratings', 'num_ratings', 'student',
                          'articles', 'exercise', 'downloadable_resources']
for col in kolom_numerik_nullable:
    if col in df_gabungan.columns:
        df_gabungan[col] = df_gabungan[col].fillna(0)

print(f"Dimensi setelah dropna pada kolom wajib: {df_gabungan.shape}")
print("\nMissing values setelah pembersihan:")
print(df_gabungan.isnull().sum())

Dimensi setelah dropna pada kolom wajib: (94669, 16)

Missing values setelah pembersihan:
course_title              0
course_id                 0
lecturer_name             0
related_topics            0
subject                   0
level                     0
durations                 0
ratings                   0
num_ratings               0
student                   0
price                     0
last_update               0
articles                  0
exercise                  0
downloadable_resources    0
url                       0
dtype: int64


In [17]:
if 'durations' in df_gabungan.columns:
    df_gabungan['durations'] = (
        df_gabungan['durations']
        .astype(str)
        .str.replace('hours', '', case=False)
        .str.strip()
    )
    df_gabungan['durations'] = pd.to_numeric(df_gabungan['durations'], errors='coerce').fillna(0)

kolom_int = ['course_id', 'student', 'num_ratings', 'articles', 'exercise', 'downloadable_resources']
for col in kolom_int:
    if col in df_gabungan.columns:
        df_gabungan[col] = pd.to_numeric(df_gabungan[col], errors='coerce').fillna(0).astype(int)

if 'price' in df_gabungan.columns:
    df_gabungan['price'] = (
        df_gabungan['price']
        .astype(str)
        .str.replace('Rp', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.replace('Free', '0', case=False, regex=False)
        .str.strip()
    )
    df_gabungan['price'] = pd.to_numeric(df_gabungan['price'], errors='coerce').fillna(0).astype(int)

print("Pembersihan numerik selesai!")
print(df_gabungan[['durations', 'price', 'course_id', 'student', 'num_ratings']].dtypes)

Pembersihan numerik selesai!
durations      float64
price            int64
course_id        int64
student          int64
num_ratings      int64
dtype: object


In [18]:
if 'lecturer_name' in df_gabungan.columns:
    df_gabungan['lecturer_name'] = df_gabungan['lecturer_name'].astype(str).str.strip()
    print("Contoh nilai lecturer_name:")
    print(df_gabungan['lecturer_name'].head(5).tolist())
    print(f"\nTotal unique lecturer: {df_gabungan['lecturer_name'].nunique()}")

Contoh nilai lecturer_name:
['GameDev.tv Team, Rick Davidson, Ahmed Nassef', 'Richard Allbert, Martyna Olivares', 'Cobra Code', 'Unity Alex Dev', 'Richard Allbert, Martyna Olivares']

Total unique lecturer: 22391


In [19]:
topic_mapping = {}

if os.path.exists(nama_file_txt):
    with open(nama_file_txt, 'r', encoding='utf-8') as file:
        baris_teks = file.readlines()

    for baris in baris_teks:
        baris = baris.strip()
        if '.' in baris:
            bagian = baris.split('.', 1)
            try:
                id_topik = int(bagian[0].strip())
                nama_topik = bagian[1].strip()
                topic_mapping[nama_topik] = id_topik
            except ValueError:
                continue
    print(f"Berhasil memuat {len(topic_mapping)} topik dari file {nama_file_txt}")
else:
    print(f"Peringatan: File '{nama_file_txt}' tidak ditemukan di direktori!")

Berhasil memuat 404 topik dari file udemy_list_topics.txt


In [20]:
if 'level' in df_gabungan.columns:
    level_mapping = {'All Levels': 4, 'Beginner': 1, 'Intermediate': 2, 'Expert': 3}
    df_gabungan['level'] = df_gabungan['level'].map(level_mapping).fillna(0).astype(int)
    print("Encoding level selesai:", df_gabungan['level'].value_counts().to_dict())

if len(topic_mapping) > 0:
    
    if 'subject' in df_gabungan.columns:
        df_gabungan['subject'] = df_gabungan['subject'].map(topic_mapping).fillna(0).astype(int)
        print(f"\nEncoding subject selesai. Nilai unik: {df_gabungan['subject'].unique()}")

    if 'related_topics' in df_gabungan.columns:
        def encode_related_topics(teks_topik):
            if pd.isna(teks_topik):
                return ''
            topik_list = [t.strip() for t in str(teks_topik).split(',')]
            nums = [str(topic_mapping[t]) for t in topik_list if t in topic_mapping]
            return ','.join(nums)

        df_gabungan['related_topics'] = df_gabungan['related_topics'].apply(encode_related_topics)
        print(f"\nEncoding related_topics selesai.")
        print("Contoh nilai:", df_gabungan['related_topics'].head(3).tolist())

print(f"\nDimensi akhir (tetap 16 kolom): {df_gabungan.shape}")

Encoding level selesai: {4: 52871, 1: 29577, 2: 10661, 3: 1560}

Encoding subject selesai. Nilai unik: [  1   4   5   6   2  16  17  18  19  20  21  22  23  24  25  26   0   7
   8  28  29  30  31  32  33  34  36  35  37   9  38  39  40  41  10  11
  12  13  14  15  42  43  45  46  47  48  49  50  53  62  91  51  52  54
  55  56  57  59  60  61  64  65  66  67  68  69  70  71  72  73  74  75
  76  80  77  78  79  81  82  83  84  58  85  86  87  88  89  92  93  97
  94  95  96  98  99 100 101 103 102 104 203]

Encoding related_topics selesai.
Contoh nilai: ['160,1,384,57,159', '1,159,103', '385,160,386,1,159']

Dimensi akhir (tetap 16 kolom): (94669, 16)


In [21]:
print("=== Info Dataset Akhir ===")
print(f"Dimensi : {df_gabungan.shape[0]} baris, {df_gabungan.shape[1]} kolom")
print(f"Kolom   : {df_gabungan.columns.tolist()}")
print()
print("Tipe data:")
print(df_gabungan.dtypes)
print()
print("Missing values:")
print(df_gabungan.isnull().sum())
print()
print("Preview 3 baris pertama:")
df_gabungan.head(3)

=== Info Dataset Akhir ===
Dimensi : 94669 baris, 16 kolom
Kolom   : ['course_title', 'course_id', 'lecturer_name', 'related_topics', 'subject', 'level', 'durations', 'ratings', 'num_ratings', 'student', 'price', 'last_update', 'articles', 'exercise', 'downloadable_resources', 'url']

Tipe data:
course_title                  str
course_id                   int64
lecturer_name                 str
related_topics                str
subject                     int64
level                       int64
durations                 float64
ratings                   float64
num_ratings                 int64
student                     int64
price                       int64
last_update                   str
articles                    int64
exercise                    int64
downloadable_resources      int64
url                           str
dtype: object

Missing values:
course_title              0
course_id                 0
lecturer_name             0
related_topics            0
subject         

,course_title,course_id,lecturer_name,related_topics,subject,level,durations,ratings,num_ratings,student,price,last_update,articles,exercise,downloadable_resources,url
0,Complete C# Unity 2D Game Development (Updated...,258316,"GameDev.tv Team, Rick Davidson, Ahmed Nassef","160,1,384,57,159",1,4,17.5,4.8,107868,489625,169000,2/2026,2,0,8,https://www.udemy.com/course/unitycourse/
1,Jumpstart to 2D Game Development: Godot 4.5/6 ...,5348848,"Richard Allbert, Martyna Olivares","1,159,103",1,4,69.0,4.8,3870,23655,169000,3/2026,14,0,134,https://www.udemy.com/course/jumpstart-to-2d-g...
2,The Ultimate Unreal Engine 2D Game Development...,5258518,Cobra Code,"385,160,386,1,159",1,4,12.0,4.8,3214,14586,159000,3/2026,19,0,11,https://www.udemy.com/course/unreal-2d-course/


In [22]:
nama_file_baru = 'udemy_course_dataset.csv'

df_gabungan.to_csv(nama_file_baru, index=False)

print(f"Data disimpan sebagai: {nama_file_baru}")
print(f"Total: {df_gabungan.shape[0]} baris, {df_gabungan.shape[1]} kolom")

Data disimpan sebagai: udemy_course_dataset.csv
Total: 94669 baris, 16 kolom
